In [ ]:
print("Project Started")

Project Started


In [ ]:
!pip install anthropic
!pip install pydantic
!pip install pymupdf
!pip install pdfplumber

In [ ]:
import os

os.makedirs("ai-medical-report-analyzer", exist_ok=True)

print("Folder Created")

Folder Created


In [ ]:
%cd ai-medical-report-analyzer

/content/ai-medical-report-analyzer/ai-medical-report-analyzer/ai-medical-report-analyzer


In [ ]:
%%writefile schema.py

from pydantic import BaseModel, Field
from typing import List

class MedicalReport(BaseModel):

    patient_name: str

    age: int = Field(gt=0, lt=120)

    diagnosis: List[str]

Writing schema.py


In [ ]:
!ls

schema.py


In [ ]:
%%writefile extractor.py
import anthropic
import os

from dotenv import load_dotenv

load_dotenv()

# Claude Client
client = anthropic.Anthropic(
    api_key=os.getenv("YOUR-API-KEY")
)

# Tool Schema
extract_tool = {

    "name": "extract_medical_report",

    "description":
        "Extract structured medical data",

    "input_schema": {

        "type": "object",

        "properties": {

            "patient_name": {
                "type": "string"
            },

            "age": {
                "type": "number"
            },

            "diagnosis": {

                "type": "array",

                "items": {
                    "type": "string"
                }
            }
        },

        "required": [
            "patient_name",
            "age",
            "diagnosis"
        ]
    }
}

Overwriting extractor.py


In [ ]:
%%writefile validator.py
from schema import MedicalReport

def validate_report(data):

    try:

        validated = MedicalReport(**data)

        # SUCCESS CASE
        return (
            validated,
            None,
            None
        )

    except Exception as e:

        error_text = str(e)

        detected_pattern = None

        # Pattern Detection
        if "age" in error_text:

            detected_pattern = "invalid_age"

        elif "diagnosis" in error_text:

            detected_pattern = (
                "missing_diagnosis"
            )

        else:

            detected_pattern = (
                "unknown_validation_failure"
            )

        # FAILURE CASE
        return (
            None,
            error_text,
            detected_pattern
        )

Writing validator.py


In [ ]:
%%writefile retry_handler.py
# =========================================
# retry_handler.py
# =========================================

# Import Claude client and tool schema
from extractor import client, extract_tool

# Import validation function
from validator import validate_report

# Import logging function
from logger import log_error


# Maximum retry attempts
MAX_RETRIES = 3


# =========================================
# Main Retry Workflow Function
# =========================================

def extract_with_retry(medical_text):

    # =====================================
    # Initial conversation message
    # =====================================

    # Claude conversation starts here
    messages = [

        {
            "role": "user",

            # Extracted medical report text
            "content": medical_text
        }
    ]


    # =====================================
    # Retry Loop
    # =====================================

    # Retry extraction multiple times
    # if validation fails
    for attempt in range(MAX_RETRIES):

        print(f"\nAttempt {attempt + 1}")


        # =====================================
        # Claude API Call
        # =====================================

        response = client.messages.create(

            # Claude model
            model="claude-opus-4-20250514",

            # Maximum response tokens
            max_tokens=1024,

            # Tool schema definition
            tools=[extract_tool],

            # Force Claude to use tool output
            tool_choice={

                "type": "tool",

                "name": "extract_medical_report"
            },

            # Conversation history
            messages=messages
        )


        # =====================================
        # Extract Tool Output Safely
        # =====================================

        tool_output = None


        # Claude response may contain multiple blocks
        for block in response.content:

            # Find tool output block
            if block.type == "tool_use":

                # Extract structured JSON output
                tool_output = block.input

                break


        # =====================================
        # Safety Check
        # =====================================

        # Claude failed to return tool output
        if tool_output is None:

            error = "No tool output returned"

            detected_pattern = "missing_tool_output"

            print(error)

            # Log failure
            log_error(
                error,
                detected_pattern
            )

            # Retry again
            continue


        # =====================================
        # Print Extracted Output
        # =====================================

        print("\nExtracted Output:")

        print(tool_output)


        # =====================================
        # Validate Structured Output
        # =====================================

        validated, error, detected_pattern = (

            validate_report(tool_output)
        )


        # =====================================
        # SUCCESS CASE
        # =====================================

        # Validation passed
        if not error:

            print(
                f"\nValidation Successful on attempt {attempt + 1}"
            )

            # Return validated JSON
            return validated


        # =====================================
        # FAILURE CASE
        # =====================================

        print(
            f"\nValidation Failed on attempt {attempt + 1}"
        )

        print(error)

        print(
            f"Detected Pattern: {detected_pattern}"
        )


        # =====================================
        # Log Validation Failure
        # =====================================

        log_error(
            error,
            detected_pattern
        )


        # =====================================
        # Add Previous AI Response
        # =====================================

        # Add failed response into conversation
        messages.append({

            "role": "assistant",

            "content": str(tool_output)
        })


        # =====================================
        # Append Validation Feedback
        # =====================================

        # Tell Claude what went wrong
        messages.append({

            "role": "user",

            "content":
            f"""

            Validation failed.

            Detected Pattern:
            {detected_pattern}

            Fix these issues:
            {error}

            IMPORTANT:
            - Ensure age is realistic
            - Ensure diagnosis is a list
            - Return valid structured JSON
            - Follow the schema exactly

            Re-extract correctly.
            """
        })


    # =====================================
    # Final Failure After Maximum Retries
    # =====================================

    raise Exception(

        f"""
        Extraction failed after
        {MAX_RETRIES} attempts
        """
    )

Writing retry_handler.py


In [ ]:
%%writefile logger.py
import logging
import os

os.makedirs("logs", exist_ok=True)
logging.basicConfig(

    filename="logs/validation_errors.log",
    level=logging.ERROR,

    format="""
    %(asctime)s
    %(levelname)s
    %(message)s
    """
)

def log_error(error, detected_pattern):

    logging.error(
        f"""
        Pattern:
        {detected_pattern}

        Error:
        {error}
        """
    )

Writing logger.py


In [ ]:
%%writefile main.py

from pdf_reader import extract_text_from_pdf
from retry_handler import extract_with_retry

pdf_path = "sample_medical_report.pdf"

print("Reading PDF...")

text = extract_text_from_pdf(pdf_path)

print("PDF Text Extracted")

result = extract_with_retry(text)

print("\nFinal Output:\n")

print(result)

Writing main.py


In [ ]:
%%writefile pdf_reader.py
import pdfplumber


def extract_text_from_pdf(pdf_path):

    extracted_text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            text = page.extract_text()

            if text:

                extracted_text += text + "\n"

    return extracted_text

Writing pdf_reader.py


In [ ]:
%%writefile schema.py
from pydantic import BaseModel, Field
from typing import List

class MedicalReport(BaseModel):

    patient_name: str

    age: int = Field(
        gt=0,
        lt=120
    )

    diagnosis: List[str]



Overwriting schema.py


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving sample_medical_report.pdf to sample_medical_report.pdf


In [ ]:
!python main.py

Reading PDF...
PDF Text Extracted

Attempt 1

Extracted Output:
{'patient_name': 'John Doe', 'age': 45, 'diagnosis': ['Diabetes', 'Hypertension']}

Validation Successful on attempt 1

Final Output:

patient_name='John Doe' age=45 diagnosis=['Diabetes', 'Hypertension']


In [ ]:
!ls

extractor.py  logs     pdf_reader.py  retry_handler.py		 schema.py
logger.py     main.py  __pycache__    sample_medical_report.pdf  validator.py
